# Phase 2 — consent-based enrollment
## Goal
Store one template per voluntary adult pseudonym. Normalize each of 3–5 distinct captures, require pairwise consistency, average and re-normalize. Templates and capture hashes are authenticated-encrypted; no raw images are saved by enrollment.

This checkout is under **OneDrive**. Before storing real biometrics, pause/exclude cloud synchronization or relocate the research checkout to a genuinely local folder, restrict NTFS permissions, and keep the key in a user-controlled secret source. Deleting a local file cannot revoke synced copies/backups.

## Setup

In [ ]:
from pathlib import Path
import sys, json, os, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/phase2_verification.yaml').exists())
sys.path.insert(0,str(ROOT/'src'))
os.environ['MPLCONFIGDIR'] = str(ROOT/'.cache/matplotlib')
from face_verification import *
config = load_config(root=ROOT)
REPORTS = ROOT/'reports/phase2'
# Real faces are opt-in. Set these manually only after confirming each participant's consent.
RUN_REAL_FACES = False
ADULT_CONSENT = False
IDENTITY_CONSENT = False
PRIVATE_MANIFEST = ROOT/'data/private/consent_manifest.json'
print('Interpreter:',sys.executable)
print('Real-face workflow enabled:',RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT)


PRIVATE_INPUTS = Path(os.environ.get('FACEATTEND_PRIVATE_INPUTS', str(Path.home()/'AppData/Local/FaceAttend/phase2/private')))
# Explicit executor opt-in; ordinary Run All keeps all defaults false.
if os.environ.get('FACEATTEND_CONSENTED_SMOKE') == '1':
    supplied = json.loads((PRIVATE_INPUTS/'input_manifest.json').read_text())
    assert supplied['adult_identity_verification_consent_confirmed_by_user'] is True
    assert supplied['research_terms_confirmed_by_user'] is True
    RUN_REAL_FACES = ADULT_CONSENT = IDENTITY_CONSENT = True
print('Consented three-image smoke enabled:',RUN_REAL_FACES)


## Synthetic enrollment and encrypted storage check
This uses deliberately injected synthetic vectors and no face inference. An ephemeral key is kept only in process memory; no key is printed or saved. The temporary synthetic store is deleted when the cell ends.

In [ ]:
from tempfile import TemporaryDirectory
from cryptography.fernet import Fernet
from types import SimpleNamespace
from face_verification.schemas import FaceResult
import numpy as np
with TemporaryDirectory(dir=ROOT/'.cache/tmp') as temporary:
    synthetic_store = TemplateStore(Path(temporary)/'templates',key=Fernet.generate_key())
    fake_models = SimpleNamespace(config=config,model_id='synthetic_unit_test',embedding_dim=3)
    images = [np.full((200,200,3),100+i,np.uint8) for i in range(3)]
    fake_detect = lambda image,models:FaceResult(Status.OK,aligned=image)
    fake_embed = lambda face,models:np.array([1.,0.,0.])
    missing_consent = enroll_user('volunteer_001',images,synthetic_store,fake_models,detector=fake_detect,embedder=fake_embed)
    result = enroll_user('volunteer_001',images,synthetic_store,fake_models,adult_consent=True,identity_consent=True,detector=fake_detect,embedder=fake_embed)
    duplicate = enroll_user('volunteer_001',images,synthetic_store,fake_models,adult_consent=True,identity_consent=True)
    encrypted = next(synthetic_store.directory.glob('*.fernet')).read_bytes()
    print({'synthetic_only':True,'missing_consent':missing_consent.status.value,'enrollment':result.status.value,
        'duplicate_id':duplicate.status.value,'norm':float(np.linalg.norm(synthetic_store.load('volunteer_001')['embedding'])),
        'plaintext_embedding_field_present':b'embedding' in encrypted,'plaintext_id_present':b'volunteer_001' in encrypted})
    print('Participant removal:',delete_enrollment('volunteer_001',synthetic_store).status.value)

## Real enrollment — manual opt-in
The key must exist in `FACEATTEND_TEMPLATE_KEY` before this cell. Never paste it into a saved cell. The manifest contains pseudonyms, adult identity-verification consent, image paths, and capture-session labels. Keep it private. Do not overwrite duplicate enrollment; delete then recollect and re-enroll.

In [ ]:
if RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT:
    import cv2
    models = initialize_models(config)
    with TemporaryDirectory(dir=ROOT/'.cache/tmp') as temporary:
        empty_store = TemplateStore(Path(temporary),key=Fernet.generate_key(),retention_days=config.retention_days)
        for index,path in enumerate(sorted((PRIVATE_INPUTS/'captures').glob('*_input.jpeg')),1):
            user_id = path.name.removesuffix('_input.jpeg')
            result = enroll_user(user_id,[cv2.imread(str(path))],empty_store,models,adult_consent=True,identity_consent=True)
            print({'input_index':index,'status':result.status.value,'message':result.message})
            assert result.status == Status.INSUFFICIENT_DATA
        assert not list(Path(temporary).glob('*.fernet'))
else:
    print('NOT_RUN: explicit private-image consent switch is disabled.')

## Checks and next steps
The real one-photo-per-person attempts return INSUFFICIENT_DATA before embedding/storage. No enrollment records or keys are persisted. Provide 3–5 distinct clear captures per person, then enroll under their selected pseudonym using a user-controlled key. The synthetic encrypted round-trip, duplicate-ID and deletion demonstration remains separate from actual participant enrollment.

Private supplied inputs are outside OneDrive and the repository with a user-only NTFS ACL. Raw-file retention is the operator's responsibility; the template API's configurable 30-day expiry does not delete source photos or backups.

## Latest multi-image encrypted enrollment smoke

This section reads the executed ten-image retest artifact. The initial three-image experiment above is retained as history. User-supplied labels are represented by pseudonyms; no portraits or vectors are displayed. The user confirmed all ten captures are from one session. Enrollment uses photos 1/2/4 and 7/8/9; probes use 3/5 and 6/10, selected before score inspection. The encrypted enrollment smoke used a process-only key and removed all templates afterward.

These results establish implementation behavior, not a calibrated matching decision, independent-session accuracy, liveness, FMR/FNMR reliability or production readiness.

In [ ]:
batch = json.loads((REPORTS/'batch02_results.json').read_text())
print(json.dumps({'enrollment':batch['enrollment_smoke'],'removal':batch['template_deletion'],
    'templates_remaining':batch['templates_remaining'],'keys_persisted':batch['keys_persisted']},indent=2))
assert all(e['status']=='OK' and e['encrypted_roundtrip'] for e in batch['enrollment_smoke'])
assert batch['templates_remaining']==0 and not batch['keys_persisted']

## Latest single-subject cross-session diagnostic

This section reads `batch03_results.json`, produced by an actual fresh-process GPU run. The user supplied one subject's ten photos and confirmed every photo came from a separate session; exact dates were not supplied. Enrollment photos 1/2/7 and probe photos 3/4/5/6/8/9/10 were fixed before inspecting scores. Photos 6/10 fail the unchanged size/clipping gate. No rejected image is embedded.

The three-image encrypted enrollment, duplicate-ID rejection and deletion passed. Five valid probes have diagnostic genuine scores 0.529–0.835; ten comparisons with explicitly labeled prior controls have scores 0.022–0.268. Session independence is user-confirmed for the new subject; cross-subject chronology is unconfirmed. Comparisons share subjects/templates, so the counts are dependent evidence, not independent population trials. All operational verification attempts refuse with THRESHOLD_NOT_CALIBRATED. No production threshold, FMR/FNMR, liveness or attendance claim is issued.

Names, original images and vectors remain out of public notebook outputs. Test templates and the process-only key were removed. Earlier cohorts above are preserved as history.

In [ ]:
latest = json.loads((REPORTS/'batch03_results.json').read_text())
print(json.dumps({'enrollment':latest['enrollment_smoke'],'prior_control_enrollments':latest['prior_control_enrollments'],'deletion':latest['template_deletion'],'templates_remaining':latest['templates_remaining'],'keys_persisted':latest['keys_persisted']},indent=2))
assert latest['enrollment_smoke']['status']=='OK' and latest['templates_remaining']==0